In [1]:
import torch as t
import torch.nn as nn
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings("ignore")

In [2]:
%load_ext autoreload
%autoreload 2

from data.featurize import SolvationDataset
from model.mpnn import MPNNModel
from model.gcn import GCNModel
from model.gat import GATModel

In [3]:
class EarlyStopper:
    def __init__(self, patience=1, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.min_validation_loss = float("inf")

    def early_stop(self, validation_loss):
        if validation_loss < self.min_validation_loss:
            self.min_validation_loss = validation_loss
            self.counter = 0
        elif validation_loss > (self.min_validation_loss + self.min_delta):
            self.counter += 1
            if self.counter >= self.patience:
                return True
        return False

In [4]:
def train_model(model, train_loader, val_loader, epochs=100, lr=0.001):    
    device = t.device("cuda" if t.cuda.is_available() else "cpu")
    model = model.to(device)
    
    optimizer = t.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    criterion = nn.MSELoss()
    
    scheduler = t.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)
    
    early_stopper = EarlyStopper(patience=3, min_delta=10)
    
    history = {"train_loss": [], "val_loss": [], "val_mae": []}
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        train_count = 0
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            
            try:
                out = model(batch)
                
                if t.isnan(out).any() or t.isinf(out).any():
                    print(f"Warning: NaN/Inf detected in model output at epoch {epoch+1}")
                    continue
                
                loss = criterion(out, batch.y)
                
                if t.isnan(loss) or t.isinf(loss):
                    print(f"Warning: NaN/Inf detected in loss at epoch {epoch+1}")
                    continue
                
                loss.backward()
                
                # Gradient clipping
                max_grad_norm = 1.0
                t.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                
                optimizer.step()
                train_loss += loss.item()
                train_count += 1
            except Exception as e:
                print(f"Error in training batch: {e}")
                continue
        
        train_loss = train_loss / train_count if train_count > 0 else float("nan")
        
        model.eval()
        val_loss = 0
        val_mae = 0
        val_count = 0
        with t.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                try:
                    out = model(batch)
                    
                    if t.isnan(out).any() or t.isinf(out).any():
                        continue
                    
                    loss = criterion(out, batch.y)
                    mae = t.abs(out - batch.y).mean()
                    
                    if not (t.isnan(loss) or t.isinf(loss)):
                        val_loss += loss.item()
                        val_mae += mae.item()
                        val_count += 1
                except Exception as e:
                    print(f"Error in validation batch: {e}")
                    continue
        
        val_loss = val_loss / val_count if val_count > 0 else float("nan")
        val_mae = val_mae / val_count if val_count > 0 else float("nan")
        
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_mae"].append(val_mae)
        
        scheduler.step(val_loss)
        
        if early_stopper.early_stop(val_loss):
            break
        
        if (epoch + 1) % 1 == 0:
            print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val MAE: {val_mae:.4f}")
    
    return history

In [9]:
# dataset assumed to contain: solute SMILES, solvent SMILES, free energy
dataset = SolvationDataset("data/CombiSolv-QM-sample-1.csv")
dataset_list = [dataset[i] for i in range(len(dataset))]
train_data, val_data = train_test_split(
    dataset_list, 
    test_size=0.2, 
    random_state=42, 
    shuffle=True
)

train_loader = DataLoader(train_data, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_data, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

print(f"Dataset size: {len(dataset)}, Train: {len(train_data)}, Val: {len(val_data)}")

sample = dataset[0]
node_feat_dim = sample.x.size(1)
edge_feat_dim = sample.edge_attr.size(1) if sample.edge_attr.size(0) > 0 else 4

print(f"Node feature dimension: {node_feat_dim}")
print(f"Edge feature dimension: {edge_feat_dim}")

Dataset size: 26, Train: 20, Val: 6
Node feature dimension: 8
Edge feature dimension: 4


In [11]:
%%time
# Train MPNN model
mpnn_model = MPNNModel(node_feat_dim, edge_feat_dim, hidden_dim=48, num_layers=2)
mpnn_history = train_model(mpnn_model, train_loader, val_loader, epochs=50, lr=0.002)
t.save(mpnn_model.state_dict(), "checkpoints/mpnn_solvation_model.pt")
print(f"MPNN Final Val MAE: {mpnn_history["val_mae"][-1]:.4f}")

Epoch 1/50 - Train Loss: 56.9852, Val Loss: 27.5789, Val MAE: 4.9362
Epoch 2/50 - Train Loss: 52.4030, Val Loss: 25.5531, Val MAE: 4.7266
Epoch 3/50 - Train Loss: 47.8199, Val Loss: 21.8387, Val MAE: 4.3158
Epoch 4/50 - Train Loss: 46.1317, Val Loss: 15.6957, Val MAE: 3.5331
Epoch 5/50 - Train Loss: 34.6914, Val Loss: 7.6407, Val MAE: 2.2250
Epoch 6/50 - Train Loss: 22.6334, Val Loss: 3.3851, Val MAE: 1.6598
Epoch 7/50 - Train Loss: 11.5479, Val Loss: 18.3551, Val MAE: 3.8913
Epoch 8/50 - Train Loss: 15.0928, Val Loss: 58.7594, Val MAE: 7.4530
MPNN Final Val MAE: 8.8688
CPU times: user 3.69 s, sys: 698 ms, total: 4.39 s
Wall time: 2.74 s


In [12]:
%%time
# Train GCN model
gcn_model = GCNModel(node_feat_dim, hidden_dim=64, num_layers=3)
gcn_history = train_model(gcn_model, train_loader, val_loader, epochs=100, lr=0.001)
t.save(gcn_model.state_dict(), "checkpoints/gcn_solvation_model.pt")
print(f"GCN Final Val MAE: {gcn_history["val_mae"][-1]:.4f}")

Epoch 1/100 - Train Loss: 50.9618, Val Loss: 26.3642, Val MAE: 4.8116
Epoch 2/100 - Train Loss: 43.4566, Val Loss: 25.8604, Val MAE: 4.7590
Epoch 3/100 - Train Loss: 36.9486, Val Loss: 25.2848, Val MAE: 4.6981
Epoch 4/100 - Train Loss: 49.9362, Val Loss: 24.4988, Val MAE: 4.6137
Epoch 5/100 - Train Loss: 58.6090, Val Loss: 23.4120, Val MAE: 4.4944
Epoch 6/100 - Train Loss: 48.7901, Val Loss: 22.0291, Val MAE: 4.3378
Epoch 7/100 - Train Loss: 44.2786, Val Loss: 20.3165, Val MAE: 4.1357
Epoch 8/100 - Train Loss: 49.0436, Val Loss: 18.1546, Val MAE: 3.8655
Epoch 9/100 - Train Loss: 53.0030, Val Loss: 15.5520, Val MAE: 3.5127
Epoch 10/100 - Train Loss: 44.7249, Val Loss: 12.5103, Val MAE: 3.0492
Epoch 11/100 - Train Loss: 37.4401, Val Loss: 9.2094, Val MAE: 2.4546
Epoch 12/100 - Train Loss: 21.6633, Val Loss: 6.0419, Val MAE: 1.9435
Epoch 13/100 - Train Loss: 21.6841, Val Loss: 3.7215, Val MAE: 1.4531
Epoch 14/100 - Train Loss: 20.4633, Val Loss: 3.4637, Val MAE: 1.6884
Epoch 15/100 - Trai

In [13]:
%%time
# Train GAT model
gat_model = GATModel(node_feat_dim, hidden_dim=64, num_layers=3, heads=4)
gat_history = train_model(gat_model, train_loader, val_loader, epochs=100, lr=0.001)
t.save(gat_model.state_dict(), "checkpoints/gat_solvation_model.pt")
print(f"GAT Final Val MAE: {gat_history["val_mae"][-1]:.4f}")

Epoch 1/100 - Train Loss: 57.1261, Val Loss: 27.8107, Val MAE: 4.9596
Epoch 2/100 - Train Loss: 67.1829, Val Loss: 26.8009, Val MAE: 4.8568
Epoch 3/100 - Train Loss: 56.2757, Val Loss: 25.6181, Val MAE: 4.7334
Epoch 4/100 - Train Loss: 66.1075, Val Loss: 23.8566, Val MAE: 4.5436
Epoch 5/100 - Train Loss: 57.9572, Val Loss: 21.5140, Val MAE: 4.2780
Epoch 6/100 - Train Loss: 35.7613, Val Loss: 18.5671, Val MAE: 3.9185
Epoch 7/100 - Train Loss: 39.6789, Val Loss: 15.0846, Val MAE: 3.4456
Epoch 8/100 - Train Loss: 45.3714, Val Loss: 11.1837, Val MAE: 2.8233
Epoch 9/100 - Train Loss: 32.3428, Val Loss: 7.3096, Val MAE: 2.1715
Epoch 10/100 - Train Loss: 24.1890, Val Loss: 4.1809, Val MAE: 1.5433
Epoch 11/100 - Train Loss: 14.2123, Val Loss: 3.3350, Val MAE: 1.6380
Epoch 12/100 - Train Loss: 13.7160, Val Loss: 7.4107, Val MAE: 2.3580
Epoch 13/100 - Train Loss: 11.4604, Val Loss: 20.7368, Val MAE: 4.1862
Epoch 14/100 - Train Loss: 20.9075, Val Loss: 38.2591, Val MAE: 5.9200
GAT Final Val MAE: 